In [11]:
# ----------------------------
#  1. Load and inspectthe data
# ----------------------------

# import libraries
import pandas as pd
import numpy as np

# load dataset
df = pd.read_csv('../Data/Raw/sales_raw.csv')

# inspect the dataset
print(df.head())
print(df.info())
print(df.describe())
print(df.columns)

  STRUCTURE           STRUCTURE_ID  \
0  dataflow  ESTAT:STS_TRTU_A(1.0)   
1  dataflow  ESTAT:STS_TRTU_A(1.0)   
2  dataflow  ESTAT:STS_TRTU_A(1.0)   
3  dataflow  ESTAT:STS_TRTU_A(1.0)   
4  dataflow  ESTAT:STS_TRTU_A(1.0)   

                                      STRUCTURE_NAME freq Time frequency  \
0  Turnover and volume of sales in wholesale and ...    A         Annual   
1  Turnover and volume of sales in wholesale and ...    A         Annual   
2  Turnover and volume of sales in wholesale and ...    A         Annual   
3  Turnover and volume of sales in wholesale and ...    A         Annual   
4  Turnover and volume of sales in wholesale and ...    A         Annual   

  indic_bt Business trend indicator nace_r2  \
0   NETTUR             Net turnover     G47   
1   NETTUR             Net turnover     G47   
2   NETTUR             Net turnover     G47   
3   NETTUR             Net turnover     G47   
4   NETTUR             Net turnover     G47   

  Statistical classification of

In [12]:
# =======================
# 2. filter indicators, drop and rename columns
# =======================


# filter indicators
df = df[df["nace_r2"].isin(["G47", "G47_NF_HLTH", "G476"])]

# keep only the needed columns ["geo", "TIME_PERIOD", "OBS_VALUE", "unit", "nace_r2"]
df = df[["geo", "TIME_PERIOD", "OBS_VALUE", "unit", "nace_r2"]]

# rename columns
df = df.rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "sales_value",
    "unit": "sales_unit",
    "nace_r2": "indicator"
})

# show all unique indicators
print(df["indicator"].unique())

# check data types
print(df.dtypes)


<StringArray>
['G47', 'G476', 'G47_NF_HLTH']
Length: 3, dtype: str
country            str
year             int64
sales_value    float64
sales_unit         str
indicator          str
dtype: object


In [13]:
# ========================
# 5. clean missing values and douplicates, group by year and mean the sales values, interpolate missing values
# ========================
# ensure numeric
df["sales_value"] = pd.to_numeric(df["sales_value"], errors="coerce")

# remove invalid values only
df = df[df["sales_value"] > 0]



In [14]:
# ==========================
# 5. Sort for merging + time series
# ==========================

df = df.groupby(
    ["country", "year", "indicator"],
    as_index=False
)["sales_value"].mean()


df = df.sort_values(["country", "indicator", "year"])


In [15]:
# =====================
# 6. save and export the data
# =====================

# save cleaned dataset
df.to_csv('../Data/Processed/sales_clean.csv', index=False)


In [16]:
# =============================
# Sales dataset information
# =============================

# check unique countries
print("Unique countries:", df["country"].nunique())

# check unique years
print("Unique years:", df["year"].nunique())

# check unique indicators (VERY important for sales)
print("Unique indicators:", df["indicator"].nunique())
print("Indicators:", df["indicator"].unique())

# total rows
print("Total rows:", len(df))

# year range
print("Year range:", df["year"].min(), "-", df["year"].max())

# sales value range
print("Sales value range:", df["sales_value"].min(), "-", df["sales_value"].max())

Unique countries: 41
Unique years: 35
Unique indicators: 3
Indicators: <StringArray>
['G47', 'G476', 'G47_NF_HLTH']
Length: 3, dtype: str
Total rows: 2186
Year range: 1991 - 2025
Sales value range: 7.383333333333333 - 330.96666666666664
